In [5]:

import time
import warnings
from pathlib import Path

import pandas as pd
from tqdm import tqdm

from nba_api.stats.endpoints import CommonTeamRoster
from nba_api.stats.static import teams as nba_teams

warnings.filterwarnings("ignore")

SEASONS = ["2022-23", "2023-24", "2024-25"]
REQUEST_DELAY = 0.5
OUTPUT_DIR = Path(".")

In [2]:
def height_to_inches(height_str: str) -> float | None:
    """Convert "6-7" → 79.0 inches. Returns None if unparseable."""
    try:
        feet, inches = height_str.split("-")
        return int(feet) * 12 + int(inches)
    except Exception:
        return None


def fetch_roster(team_id: int, season: str) -> pd.DataFrame | None:
    try:
        roster = CommonTeamRoster(
            team_id=team_id,
            season=season,
            league_id_nullable="00",
        )
        df = roster.get_data_frames()[0]
        return df if not df.empty else None
    except Exception:
        return None

In [3]:
def run():

    all_teams = nba_teams.get_teams()
    all_frames = []

    for season in SEASONS:
        for team in tqdm(all_teams, desc=season, unit="team"):
            df = fetch_roster(team["id"], season)
            if df is not None:
                df["SEASON"] = season
                all_frames.append(df)
            time.sleep(REQUEST_DELAY)

    if not all_frames:
        return

    combined = pd.concat(all_frames, ignore_index=True)

    # Parse height to inches
    combined["HEIGHT_INCHES"] = combined["HEIGHT"].apply(height_to_inches)

    # Deduplicate — players who changed teams mid-season appear on multiple rosters;
    # keep the row with the most recent team (last appearance in the data)
    combined = combined.drop_duplicates(subset=["PLAYER_ID", "SEASON"], keep="last")

    combined.to_csv("nba_player_attributes.csv", index=False)



In [6]:
run()

2024-25: 100%|██████████| 30/30 [00:16<00:00,  1.85team/s]
